# Задача 5. PageRank

## Задание 1 Масштабирование на большой граф

- Загрузите граф web-Stanford (~280K узлов) из коллекции SNAP или аналогичный крупный разреженный граф (например, из SuiteSparse Matrix Collection).
- Реализуйте оптимизированную версию PageRank, способную работать с ограниченными ресурсами памяти. Возможные подходы:
    - Использование dtype=dtypes.FP32.
    - Блочная обработка графа (например, через gb.Matrix.ss.split()).
- Проведите бенчмаркинг:
    - Измерьте время выполнения и пиковое потребление памяти.
    - Сравните результаты с базовой последовательной реализацией по критериям: число итераций до сходимости и максимальная разница в рангах.

### Решение

Импорт и загрузка графа

In [15]:
import time
import tracemalloc
import urllib.request
import gzip
import os
import numpy as np
from scipy.sparse import csr_matrix
import graphblas as gb
from graphblas import dtypes, semiring, unary, binary, monoid, Matrix, Vector

import random
import gc
import scipy.io

In [16]:
# Скачиваем граф
def download_web_stanford(path="web-Stanford.txt.gz"):
    url = "https://snap.stanford.edu/data/web-Stanford.txt.gz"
    if not os.path.exists(path):
        print(f"Скачиваем {url} ...")
        urllib.request.urlretrieve(url, path)
        print("Готово.")
    return path

# Читаем граф из файла SNAP
def load_edges(path):
    rows, cols = [], []
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue
            u, v = map(int, line.split())
            rows.append(u)
            cols.append(v)
    return rows, cols

# Перенумеровываем вершины с нуля подряд
def remap_nodes(rows, cols):
    all_nodes = sorted(set(rows) | set(cols))
    mapping = {old: new for new, old in enumerate(all_nodes)}
    rows = [mapping[r] for r in rows]
    cols = [mapping[c] for c in cols]
    n = len(all_nodes)
    return rows, cols, n

Базовая реализация 

In [17]:
def pagerank_baseline(rows, cols, n, damping=0.85, tol=1e-6, max_iter=100):

    tracemalloc.start()
    t0 = time.perf_counter()

    # Строим матрицу: A[dst, src] = 1
    data = np.ones(len(rows), dtype=np.float64)
    A = csr_matrix((data, (cols, rows)), shape=(n, n))

    # Нормируем по исходящей степени (столбцы)
    out_deg = np.array(A.sum(axis=0)).flatten() 
    out_deg[out_deg == 0] = 1   

    # Нормируем каждый столбец
    inv_deg = 1.0 / out_deg

    A = A.multiply(inv_deg) 

    r = np.full(n, 1.0 / n, dtype=np.float64)
    iters = 0
    for i in range(max_iter):
        r_new = damping * A.dot(r) + (1 - damping) / n
        diff = np.abs(r_new - r).max()
        r = r_new
        iters += 1
        if diff < tol:
            break

    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return r, iters, elapsed, peak / 1024**2

Оптимизированная реализация PageRank на python-graphblas с dtype=FP32

In [18]:
def pagerank_graphblas_fp32(rows, cols, n, damping=0.85, tol=1e-6, max_iter=100):

    tracemalloc.start()
    t0 = time.perf_counter()

    # Создаём матрицу FP32
    rows_arr = np.array(rows, dtype=np.uint64)
    cols_arr = np.array(cols, dtype=np.uint64)
    vals_arr = np.ones(len(rows), dtype=np.float32)

    # A[dst, src] = 1.0
    A = Matrix.from_coo(
        cols_arr, rows_arr, vals_arr,
        dtype=dtypes.FP32,
        nrows=n, ncols=n
    )

    col_sums = A.reduce_columnwise(gb.monoid.plus).new()

    # Нормируем: делим каждый столбец src на col_sums[src]
    inv_deg = col_sums.dup()
    inv_deg << gb.unary.minv(col_sums)

    # Применяем по столбцам: A[i,j] *= inv_deg[j]
    D_inv = Matrix.from_coo(
        np.arange(n, dtype=np.uint64),
        np.arange(n, dtype=np.uint64),
        inv_deg.to_dense(fill_value=0.0),
        dtype=dtypes.FP32,
        nrows=n, ncols=n
    )

    # A_norm = A @ D_inv  => A_norm[i,j] = A[i,k]*D_inv[k,j] = A[i,j] * inv_deg[j]
    A_norm = (A @ D_inv).new(dtype=dtypes.FP32)

    # Вектор рангов
    r = Vector.from_dense(
        np.full(n, 1.0 / n, dtype=np.float32)
    )

    teleport = (1.0 - damping) / n

    iters = 0
    for _ in range(max_iter):
        r_new = (A_norm @ r).new(dtype=dtypes.FP32)
        r_new *= damping
        r_new += teleport

        # Проверка сходимости: max |r_new - r|
        diff_vec = (r_new - r).new()
        diff_vec << gb.unary.abs(diff_vec)
        diff = diff_vec.reduce(gb.monoid.max).new().value

        r = r_new
        iters += 1
        if diff < tol:
            break

    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    ranks = r.to_dense(fill_value=0.0)
    return ranks, iters, elapsed, peak / 1024**2

Реализация блочной обработки графа

In [19]:
def pagerank_graphblas_blocked(rows, cols, n, damping=0.85, tol=1e-6, max_iter=100,
                                block_size=50_000):

    tracemalloc.start()
    t0 = time.perf_counter()

    rows_arr = np.array(rows, dtype=np.uint64)
    cols_arr = np.array(cols, dtype=np.uint64)
    vals_arr = np.ones(len(rows), dtype=np.float32)

    
    A = Matrix.from_coo(
        cols_arr, rows_arr, vals_arr,
        dtype=dtypes.FP32, nrows=n, ncols=n
    )
    col_sums = A.reduce_columnwise(gb.monoid.plus).new()
    inv_deg = gb.unary.minv(col_sums).new()

    D_inv = Matrix.from_coo(
        np.arange(n, dtype=np.uint64),
        np.arange(n, dtype=np.uint64),
        inv_deg.to_dense(fill_value=0.0),
        dtype=dtypes.FP32, nrows=n, ncols=n
    )
    A_norm = (A @ D_inv).new(dtype=dtypes.FP32)

    # Разбиваем матрицу на блоки по строкам и столбцам
    num_blocks = (n + block_size - 1) // block_size
    row_sizes = [block_size] * (num_blocks - 1) + [n - block_size * (num_blocks - 1)]
    col_sizes = row_sizes

    # split
    blocks = A_norm.ss.split([row_sizes, col_sizes])

    r = Vector.from_dense(np.full(n, 1.0 / n, dtype=np.float32))
    teleport = (1.0 - damping) / n

    iters = 0
    for _ in range(max_iter):
        r_new_dense = np.zeros(n, dtype=np.float32)

        # Блочное умножение: для каждого блока строк
        row_offset = 0
        for bi in range(num_blocks):
            col_offset = 0
            partial = np.zeros(row_sizes[bi], dtype=np.float32)
            for bj in range(num_blocks):
                blk = blocks[bi][bj]
                if blk is not None and blk.nvals > 0:
                    # Срез вектора r для этого блока столбцов
                    r_sub = r[col_offset : col_offset + col_sizes[bj]].new()
                    prod = (blk @ r_sub).new()
                    partial += prod.to_dense(fill_value=0.0)
                col_offset += col_sizes[bj]
            r_new_dense[row_offset : row_offset + row_sizes[bi]] = partial
            row_offset += row_sizes[bi]

        r_new_dense = damping * r_new_dense + teleport
        r_new = Vector.from_dense(r_new_dense)

        diff = np.abs(r_new_dense - r.to_dense(fill_value=0.0)).max()
        r = r_new
        iters += 1
        if diff < tol:
            break

    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return r.to_dense(fill_value=0.0), iters, elapsed, peak / 1024**2

Сравнение методов

In [20]:
def print_benchmark(name, iters, elapsed, peak_mb):
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Итераций до сходимости : {iters}")
    print(f"  Время выполнения       : {elapsed:.2f} сек")
    print(f"  Пик памяти             : {peak_mb:.1f} МБ")

Запуск

In [21]:
def main():
    # --- Загрузка графа ---
    path = download_web_stanford()
    print("Читаем рёбра...")
    rows, cols = load_edges(path)
    print(f"  Рёбер загружено: {len(rows):,}")

    print("Перенумеровываем вершины...")
    rows, cols, n = remap_nodes(rows, cols)
    print(f"  Вершин: {n:,}, Рёбер: {len(rows):,}")

    # --- Baseline ---
    print("\nЗапускаем Baseline (scipy FP64)...")
    r_base, it_base, t_base, mem_base = pagerank_baseline(rows, cols, n)
    print_benchmark("Baseline (scipy, FP64)", it_base, t_base, mem_base)

    # --- GraphBLAS FP32 ---
    print("\nЗапускаем GraphBLAS FP32...")
    r_fp32, it_fp32, t_fp32, mem_fp32 = pagerank_graphblas_fp32(rows, cols, n)
    print_benchmark("GraphBLAS FP32", it_fp32, t_fp32, mem_fp32)

    # --- GraphBLAS Блочная ---
    print("\nЗапускаем GraphBLAS Blocked (FP32, block=50k)...")
    r_blk, it_blk, t_blk, mem_blk = pagerank_graphblas_blocked(rows, cols, n)
    print_benchmark("GraphBLAS Blocked FP32", it_blk, t_blk, mem_blk)

    # --- Итоговая таблица ---
    print("\n\n" + "="*65)
    print(f"{'Метод':<30} {'Итер':>6} {'Время,с':>10} {'Память,МБ':>12}")
    print("="*65)
    print(f"{'Baseline (scipy FP64)':<30} {it_base:>6} {t_base:>10.2f} {mem_base:>12.1f}")
    print(f"{'GraphBLAS FP32':<30} {it_fp32:>6} {t_fp32:>10.2f} {mem_fp32:>12.1f}")
    print(f"{'GraphBLAS Blocked FP32':<30} {it_blk:>6} {t_blk:>10.2f} {mem_blk:>12.1f}")
    print("="*65)


if __name__ == "__main__":
    main()

Читаем рёбра...
  Рёбер загружено: 2,312,497
Перенумеровываем вершины...
  Вершин: 281,903, Рёбер: 2,312,497

Запускаем Baseline (scipy FP64)...

  Baseline (scipy, FP64)
  Итераций до сходимости : 41
  Время выполнения       : 0.56 сек
  Пик памяти             : 93.6 МБ

Запускаем GraphBLAS FP32...

  GraphBLAS FP32
  Итераций до сходимости : 100
  Время выполнения       : 0.71 сек
  Пик памяти             : 371.4 МБ

Запускаем GraphBLAS Blocked (FP32, block=50k)...

  GraphBLAS Blocked FP32
  Итераций до сходимости : 41
  Время выполнения       : 2.09 сек
  Пик памяти             : 230.8 МБ


Метод                            Итер    Время,с    Память,МБ
Baseline (scipy FP64)              41       0.56         93.6
GraphBLAS FP32                    100       0.71        371.4
GraphBLAS Blocked FP32             41       2.09        230.8


### Вывод

**Сходимость**

Реализация FP32 единственная, которая не сходится за 100 итераций по одной причине — ограниченная точность 32-битной арифметики. FP32 обеспечивает лишь ~7 значимых десятичных цифр против ~15 у FP64. При пороге сходимости tol=1e-6 вычисленный diff в FP32 никогда не опускается ниже этого порога из-за накопленных ошибок округления при матрично-векторных умножениях. Blocked-версия избегает этой проблемы, поскольку финальная проверка сходимости там выполняется через NumPy в FP64 (np.abs(...).max()), а не средствами GraphBLAS

**Время**

Реализация Baseline оказалась быстрее FP32 и Blocked FP32. Причин несколько: 
- GraphBLAS несёт накладные расходы на управление разреженными структурами и компиляцию операций, а scipy.sparse.csr_matrix + NumPy — зрелый, хорошо оптимизированный стек с минимальными накладными расходами
- GraphBLAS FP32 выполнил 100 итераций против 41 у Baseline, то есть фактически сделал в 2.4 раза больше работы — при пересчёте на одну итерацию GraphBLAS FP32 был бы быстрее примерно в 1.7 раза
- Blocked версия ожидаемо медленнее всех т.к. блочное разбиение через ss.split() добавляет накладные расходы на нарезку матрицы и сборку частичных результатов на каждой итерации, что перевешивает выгоду от экономии памяти

**Потребление памяти**

Baseline потребляет меньше всего памяти, поскольку scipy CSR хранит только три массива (data, indices, indptr) в компактном FP64. GraphBLAS FP32, вопреки ожиданиям, потребляет больше Blocked версии. Хранение в FP32 вдвое экономит память на значениях, но GraphBLAS добавляет значительный служебный overhead.  Blocked версия экономит память за счёт того, что в каждый момент в оперативной памяти находится только один блок матрицы, а не вся целиком

**Итог**

Ни одна из оптимизированных реализаций не достигла одновременно всех трёх целей на данном графе. GraphBLAS FP32 проигрывает по итерациям и памяти, а Blocked версия корректна по итерациям, но существенно медленнее и всё ещё тяжелее Baseline по памяти. Основная практическая ценность блочного подхода проявится на графах, не помещающихся в RAM целиком, где Baseline просто невозможен.

## Задание 2 Динамический (инкрементальный) PageRank

Реализуйте функцию быстрого обновления рангов при добавлении новых рёбер без полного пересчёта

### Решение

Вспомогательные функции

In [22]:
# Скачивание графа SNAP
def download_web_stanford(path="web-Stanford.txt.gz"):
    url = "https://snap.stanford.edu/data/web-Stanford.txt.gz"
    
    if not os.path.exists(path):
        print(f"Скачиваем {url} ...")
        urllib.request.urlretrieve(url, path)
        print("Готово.")
    else:
        print("Файл уже существует.")
        
    return path


# Универсальная загрузка графа
def load_graph(filepath: str):
    ext = os.path.splitext(filepath)[1].lower()

    # Matrix Market формат
    if ext == ".mtx":
        A = scipy.io.mmread(filepath).tocsr()
        coo = A.tocoo()
        return (
            coo.row.astype(np.int64),
            coo.col.astype(np.int64),
            A.shape[0]
        )

    # SNAP .txt.gz или обычный txt
    opener = gzip.open if filepath.endswith(".gz") else open

    src_list, dst_list = [], []
    max_node = 0

    with opener(filepath, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue

            parts = line.split()
            if len(parts) < 2:
                continue

            u, v = int(parts[0]), int(parts[1])

            src_list.append(u)
            dst_list.append(v)

            max_node = max(max_node, u, v)

    return (
        np.array(src_list, dtype=np.int64),
        np.array(dst_list, dtype=np.int64),
        max_node + 1
    )


# Использование
path = download_web_stanford()
rows, cols, n = load_graph(path)

print("Количество вершин:", n)
print("Количество рёбер:", len(rows))

# Нормализованная матрица переходов
def build_stochastic_matrix(src: np.ndarray, dst: np.ndarray, n: int,
                             dtype=dtypes.FP64) -> tuple:

    np_dtype = np.float32 if dtype == dtypes.FP32 else np.float64
    out_degree = np.bincount(src, minlength=n).astype(np.float64)
    safe_deg = np.where(out_degree > 0, out_degree, 1.0)
    data = (1.0 / safe_deg[src]).astype(np_dtype)
    P = Matrix.from_coo(
        dst.astype(np.uint64), src.astype(np.uint64), data,
        dtype=dtype, nrows=n, ncols=n,
    )
    return P, out_degree

# Полный PageRank
def full_pagerank(src: np.ndarray, dst: np.ndarray, n: int,
                  alpha=0.85, tol=1e-6, max_iter=100,
                  dtype=dtypes.FP64) -> tuple:

    np_dtype = np.float32 if dtype == dtypes.FP32 else np.float64
    P, out_degree = build_stochastic_matrix(src, dst, n, dtype=dtype)
    dangling_mask = (out_degree == 0)
    has_dangling = dangling_mask.any()

    rank_arr = np.full(n, 1.0 / n, dtype=np_dtype)
    rank = Vector.from_dense(rank_arr, dtype=dtype)
    teleport = np_dtype((1.0 - alpha) / n)

    for iteration in range(1, max_iter + 1):
        rank_arr = rank.to_dense(fill_value=np_dtype(0.0))
        if has_dangling:
            dangling_contrib = np_dtype(alpha * rank_arr[dangling_mask].sum() / n)
        else:
            dangling_contrib = np_dtype(0.0)

        new_rank = P.mxv(rank, semiring.plus_times).new(dtype=dtype)
        new_rank *= alpha
        new_rank += (dangling_contrib + teleport)

        new_arr = new_rank.to_dense(fill_value=np_dtype(0.0))
        err = float(np.abs(new_arr - rank_arr).max())
        rank = new_rank
        if err < tol:
            return rank, iteration

    return rank, max_iter

Файл уже существует.
Количество вершин: 281904
Количество рёбер: 2312497


Инкрементальный PageRank

In [23]:
def incremental_pagerank(
    A_old: Matrix,
    r_old: Vector,
    new_edges: list,
    damping: float = 0.85,
    max_iter: int = 50,
    tol: float = 1e-7,
) -> Vector:

    n = A_old.nrows
    dtype = r_old.dtype

    np_dtype = np.float32 if dtype == dtypes.FP32 else np.float64

    # Шаг 1: строим A_new = A_old + ΔA

    new_src = np.array([u for u, v in new_edges], dtype=np.uint64)
    new_dst = np.array([v for u, v in new_edges], dtype=np.uint64)
    delta_vals = np.ones(len(new_edges), dtype=np_dtype)

    delta_A = Matrix.from_coo(
        new_dst, new_src, delta_vals,
        dtype=dtype, nrows=n, ncols=n,
    )

    A_new = A_old.ewise_union(delta_A, binary.max,
                              left_default=np_dtype(0.0),
                              right_default=np_dtype(0.0)).new(dtype=dtype)

    
    # Шаг 2: пересчёт нормализации для изменённых вершин-источников
    
    # Вершины, у которых изменилась исходящая степень
    changed_sources = list(set(int(u) for u in new_src))

    # Старые и новые out_degree для изменённых вершин
    if A_old.nvals > 0:
        old_rows, old_cols, _ = A_old.to_coo()
    else:
        old_rows, old_cols = np.array([], np.uint64), np.array([], np.uint64)

    if A_new.nvals > 0:
        new_rows, new_cols, _ = A_new.to_coo()
    else:
        new_rows, new_cols = np.array([], np.uint64), np.array([], np.uint64)

    old_out_deg = np.bincount(old_cols.astype(np.int64), minlength=n).astype(np.float64)
    new_out_deg = np.bincount(new_cols.astype(np.int64), minlength=n).astype(np.float64)

    safe_new_deg = np.where(new_out_deg > 0, new_out_deg, 1.0)
    safe_old_deg = np.where(old_out_deg > 0, old_out_deg, 1.0)

    delta_M_rows, delta_M_cols, delta_M_vals = [], [], []

    changed_set = set(changed_sources)

    # Старые рёбра из изменённых источников
    for src_u, dst_v in zip(old_cols, old_rows):
        u = int(src_u)
        if u in changed_set:
            old_w = 1.0 / safe_old_deg[u]
            new_w = 1.0 / safe_new_deg[u]
            diff = new_w - old_w
            if abs(diff) > 1e-15:
                delta_M_rows.append(int(dst_v))
                delta_M_cols.append(u)
                delta_M_vals.append(np_dtype(diff))

    # Новые рёбра
    for u, v in zip(new_src, new_dst):
        u, v = int(u), int(v)
        delta_M_rows.append(v)
        delta_M_cols.append(u)
        delta_M_vals.append(np_dtype(1.0 / safe_new_deg[u]))

    if delta_M_rows:
        from collections import defaultdict
        dm_dict = defaultdict(float)
        for r, c, val in zip(delta_M_rows, delta_M_cols, delta_M_vals):
            dm_dict[(r, c)] += float(val)
        dm_rows = np.array([k[0] for k in dm_dict], dtype=np.uint64)
        dm_cols = np.array([k[1] for k in dm_dict], dtype=np.uint64)
        dm_vals = np.array(list(dm_dict.values()), dtype=np_dtype)

        delta_M = Matrix.from_coo(
            dm_rows, dm_cols, dm_vals,
            dtype=dtype, nrows=n, ncols=n,
        )
    else:
        delta_M = Matrix(dtype, nrows=n, ncols=n)


    # Шаг 3: строим P_new
    
    p_new_vals = (1.0 / safe_new_deg[new_cols]).astype(np_dtype)
    if len(new_rows) > 0:
        P_new = Matrix.from_coo(
            new_rows, new_cols, p_new_vals,
            dtype=dtype, nrows=n, ncols=n,
        )
    else:
        P_new = Matrix(dtype, nrows=n, ncols=n)


    # Шаг 4: seed возмущения d = β * ΔM^T * r_old

    r_old_arr = r_old.to_dense(fill_value=np_dtype(0.0))

    if delta_M.nvals > 0:
        d_vec = delta_M.mxv(r_old, semiring.plus_times).new(dtype=dtype)
        d_vec *= damping
    else:
        d_vec = Vector(dtype, size=n)

    d_arr = d_vec.to_dense(fill_value=np_dtype(0.0))


    # Шаг 5: power iteration для Δr

    delta_r_arr = d_arr.copy()

    for _ in range(max_iter):
        delta_r_vec = Vector.from_dense(delta_r_arr, dtype=dtype)
        if P_new.nvals > 0:
            new_delta = P_new.mxv(delta_r_vec, semiring.plus_times).new(dtype=dtype)
            new_delta *= damping
            new_delta_arr = new_delta.to_dense(fill_value=np_dtype(0.0))
        else:
            new_delta_arr = np.zeros(n, dtype=np_dtype)

        new_delta_arr += d_arr

        err = float(np.abs(new_delta_arr - delta_r_arr).max())
        delta_r_arr = new_delta_arr
        if err < tol:
            break


    # Шаг 6: r_new = r_old + Δr, нормализуем

    r_new_arr = r_old_arr + delta_r_arr
    r_new_arr = np.maximum(r_new_arr, 0.0).astype(np_dtype)
    # Нормализуем: сумма = 1
    total = r_new_arr.sum()
    if total > 0:
        r_new_arr /= total

    return Vector.from_dense(r_new_arr, dtype=dtype)

Тесты

In [26]:
def run_test(filepath: str, n_new_edges: int = 100, alpha: float = 0.85):

    print(f"Загрузка графа: {os.path.basename(filepath)}")
    src, dst, n = load_graph(filepath)
    print(f"  Вершин: {n:,}   Рёбер: {len(src):,}")
    print()

    # полный PageRank на исходном графе
    print("Шаг 1. Полный PageRank на исходном графе")
    t0 = time.perf_counter()
    r_full, iters_full = full_pagerank(src, dst, n, alpha=alpha)
    t_full = time.perf_counter() - t0
    print(f"  Время: {t_full:.3f}с   Итераций: {iters_full}")

    # Строим бинарную матрицу смежности A_old для incremental
    vals_old = np.ones(len(src), dtype=np.float64)
    A_old = Matrix.from_coo(
        dst.astype(np.uint64), src.astype(np.uint64), vals_old,
        dtype=dtypes.FP64, nrows=n, ncols=n,
    )

    # генерируем новые рёбра
    print(f"\nШаг 2. Генерация {n_new_edges} новых случайных рёбер")
    existing = set(zip(src.tolist(), dst.tolist()))
    new_edges = []
    rng = random.Random(42)
    attempts = 0
    while len(new_edges) < n_new_edges and attempts < n_new_edges * 100:
        u = rng.randint(0, n - 1)
        v = rng.randint(0, n - 1)
        if u != v and (u, v) not in existing:
            new_edges.append((u, v))
            existing.add((u, v))
        attempts += 1
    print(f"  Добавлено рёбер: {len(new_edges)}")

    # Формируем src_new, dst_new для полного пересчёта
    ne_src = np.array([u for u, v in new_edges], dtype=np.int64)
    ne_dst = np.array([v for u, v in new_edges], dtype=np.int64)
    src_new = np.concatenate([src, ne_src])
    dst_new = np.concatenate([dst, ne_dst])

    # инкрементальный PageRank
    print("\nШаг 3. Инкрементальный PageRank")
    t0 = time.perf_counter()
    r_incr = incremental_pagerank(A_old, r_full, new_edges, damping=alpha)
    t_incr = time.perf_counter() - t0
    print(f"  Время: {t_incr:.3f}с")

    # полный пересчёт на новом графе
    print("\nШаг 4. Полный PageRank на новом графе")
    t0 = time.perf_counter()
    r_full_new, iters_full_new = full_pagerank(src_new, dst_new, n, alpha=alpha)
    t_full_new = time.perf_counter() - t0
    print(f"  Время: {t_full_new:.3f}с   Итераций: {iters_full_new}")

    # сравнение
    arr_incr     = r_incr.to_dense(fill_value=0.0)
    arr_full_new = r_full_new.to_dense(fill_value=0.0)

    max_diff  = float(np.abs(arr_incr - arr_full_new).max())
    mean_diff = float(np.abs(arr_incr - arr_full_new).mean())
    speedup   = t_full_new / t_incr if t_incr > 0 else float("inf")

    print()
    print("-" * 60)
    print("  Результаты сравнения")
    print("-" * 60)
    print(f"  {'Метод':<35} {'Время':>8}  {'Итер.':>6}")
    print("  " + "-" * 52)
    print(f"  {'Полный PR (исходный граф)':<35} {t_full:>7.3f}с  {iters_full:>6}")
    print(f"  {'Инкрементальный PR':<35} {t_incr:>7.3f}с  {'—':>6}")
    print()


    print()
    # Качественная оценка точности
    if max_diff < 1e-4:
        verdict = "Отличная точность (max|Δ| < 1e-4)"
    elif max_diff < 1e-3:
        verdict = "Хорошая точность (max|Δ| < 1e-3)"
    elif max_diff < 1e-2:
        verdict = "Приемлемая точность (max|Δ| < 1e-2)"
    else:
        verdict = "Низкая точность — линеаризация не успела сойтись"
    print(f"  Оценка: {verdict}")

    return {
        "t_full": t_full, "t_incr": t_incr, "t_full_new": t_full_new,
        "max_diff": max_diff, "mean_diff": mean_diff, "speedup": speedup,
    }

Основная функция

In [27]:
def main():
    folder = r"C:\Projects\ITMO_graph"

    candidates = [
        os.path.join(folder, "web-Stanford.txt"),
        os.path.join(folder, "web-Stanford.mtx"),
        os.path.join(folder, "web_Stanford.mtx"),
    ]
    filepath = next((c for c in candidates if os.path.exists(c)), None)

    if filepath is None:
        import glob
        files = sorted(glob.glob(os.path.join(folder, "*.mtx")) +
                       glob.glob(os.path.join(folder, "*.txt")))
        filepath = files[-1] if files else None

    if filepath is None:
        print("Файл не найден.")
        return

    # Тест с разным числом новых рёбер
    for n_edges in [100, 500]:
        print("\n" + "-" * 60)
        print(f"  Тест: добавляем {n_edges} новых рёбер")
        print("-" * 60)
        run_test(filepath, n_new_edges=n_edges)
        print()


if __name__ == "__main__":
    main()


------------------------------------------------------------
  Тест: добавляем 100 новых рёбер
------------------------------------------------------------
Загрузка графа: web-Stanford.txt
  Вершин: 281,904   Рёбер: 2,312,497

Шаг 1. Полный PageRank на исходном графе
  Время: 0.651с   Итераций: 100

Шаг 2. Генерация 100 новых случайных рёбер
  Добавлено рёбер: 100

Шаг 3. Инкрементальный PageRank
  Время: 0.307с

Шаг 4. Полный PageRank на новом графе
  Время: 0.647с   Итераций: 100

------------------------------------------------------------
  Результаты сравнения
------------------------------------------------------------
  Метод                                  Время   Итер.
  ----------------------------------------------------
  Полный PR (исходный граф)             0.651с     100
  Инкрементальный PR                    0.307с       —


  Оценка: Приемлемая точность (max|Δ| < 1e-2)


------------------------------------------------------------
  Тест: добавляем 500 новых рёбер
-

### Вывод

Выполненные эксперименты показывают, что инкрементальный алгоритм PageRank действительно позволяет ускорить пересчёт рангов при изменении графа по сравнению с полным повторным вычислением. Наиболее заметный выигрыш в производительности наблюдается при малом числе добавленных рёбер, а с их увеличением, скорость выполнения стандартного и инкрементального подходов становится примерно одинакова. Это объясняется тем, что вычислительная сложность инкрементального метода растёт с увеличением объёма изменений и в пределе приближается к стоимости полного пересчёта. При этом точность инкрементального приближения остаётся стабильной во всех экспериментах